# Exploratory Data Analysis — Patient Readmission

Dataset: `domain1/temp/cleaned_dataSet.csv`
Target: `readmission_flag` (0 = not readmitted, 1 = readmitted)
`patient_id` is a reference identifier only and is excluded from all statistical/correlation/distribution analysis.

This notebook profiles the data and reports data-driven findings only. No modeling or feature engineering is performed here.

In [ ]:
%pip install -q numpy pandas matplotlib seaborn scipy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')

DATA_PATH = Path("../temp/cleaned_dataSet.csv")
df = pd.read_csv(DATA_PATH)

ID_COL = 'patient_id'
TARGET = 'readmission_flag'

NUMERIC_FEATURES = [
    'age', 'bmi', 'length_of_stay_days', 'previous_admissions_12m',
    'chronic_conditions_count', 'blood_glucose', 'cholesterol_level',
    'hemoglobin', 'creatinine', 'medications_count', 'treatment_cost',
    'patient_education_score', 'number_of_procedures', 'medication_changes_during_stay'
]

CATEGORICAL_FEATURES = [
    'gender', 'smoking_status', 'diabetes_flag', 'hypertension_flag',
    'heart_disease_flag', 'icu_admission_flag', 'emergency_admission_flag',
    'high_risk_medication_flag', 'followup_scheduled_flag',
    'discharge_destination', 'insurance_type'
]

ANALYSIS_COLS = NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TARGET]

df.shape


## 1. Data Overview

### 1.1 Shape and dtypes

In [ ]:
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
dtypes_table = df.dtypes.rename('dtype').to_frame()
dtypes_table

27 columns, 50,000 rows. Four columns (`gender`, `smoking_status`, `discharge_destination`, `insurance_type`) are stored as strings — everything else, including all the flag columns, is already int64/float64. That's why those four are the ones treated as categorical below and the rest as numeric; `patient_id` is excluded regardless.

### 1.2 Descriptive statistics — numeric features

In [ ]:
numeric_desc = df[NUMERIC_FEATURES].describe().T
numeric_desc

Most numeric features have tight, plausible ranges (e.g. `age` 18-89, `bmi` 15-41). Two things stand out: `creatinine`'s minimum is negative (-0.06), which shouldn't happen for this kind of measurement, and `treatment_cost`/`medication_changes_during_stay` have maxes far above their 75th percentile (119k vs 63k; 12 vs 3) — a right-skew that shows up again in the histograms and boxplots below.

### 1.3 Descriptive statistics — categorical/flag features

In [ ]:
categorical_desc = df[CATEGORICAL_FEATURES].describe(include='all').T
categorical_desc

`gender`, `smoking_status`, `discharge_destination` and `insurance_type` are all close to evenly split across their categories (top category is only 25-50% of rows). The binary flags are more one-sided: `followup_scheduled_flag` is 1 for ~70% of rows, while `icu_admission_flag` (~15%), `heart_disease_flag` (~20%) and `high_risk_medication_flag` (~20%) are mostly 0.

## 2. Target Variable Analysis

In [ ]:
target_counts = df[TARGET].value_counts().sort_index()
target_pct = (target_counts / len(df) * 100).round(2)
target_summary = pd.DataFrame({'count': target_counts, 'pct': target_pct})
print(target_summary)

fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=target_counts.index.astype(str), y=target_counts.values, ax=ax, palette='Set2')
ax.set_xlabel('readmission_flag')
ax.set_ylabel('count')
ax.set_title('Class distribution of readmission_flag')
for i, v in enumerate(target_counts.values):
    ax.text(i, v, str(v), ha='center', va='bottom')
plt.tight_layout()
plt.show()

Target is imbalanced: 65.3% not readmitted vs 34.7% readmitted, roughly a 2:1 split. Category-level readmission rates in Section 4 should be compared against this ~35% baseline, not 50%.

## 3. Univariate Analysis

### 3.1 Numeric feature distributions (histograms)

In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(NUMERIC_FEATURES) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()
for i, col in enumerate(NUMERIC_FEATURES):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i], color='steelblue')
    axes[i].set_title(col)
for j in range(len(NUMERIC_FEATURES), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

skew_table = df[NUMERIC_FEATURES].skew().sort_values(ascending=False).rename('skewness').to_frame()
skew_table

Skewness sits near 0 for the continuous features (`age`, `bmi`, `blood_glucose`, `cholesterol_level`, `hemoglobin`, `creatinine`, `length_of_stay_days`, `patient_education_score`) — their histograms look roughly symmetric/uniform. The count-style features are the most right-skewed: `previous_admissions_12m` (0.87), `treatment_cost` (0.78), `medication_changes_during_stay` (0.70), `chronic_conditions_count` (0.60) and `number_of_procedures` (0.59), all peaking near 0-2 with a thinning tail toward higher values.

### 3.2 Numeric feature distributions (boxplots — outlier check)

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()
for i, col in enumerate(NUMERIC_FEATURES):
    sns.boxplot(x=df[col], ax=axes[i], color='lightcoral')
    axes[i].set_title(col)
for j in range(len(NUMERIC_FEATURES), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

def iqr_outlier_count(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).sum()

outlier_counts = pd.Series({col: iqr_outlier_count(df[col]) for col in NUMERIC_FEATURES}, name='iqr_outlier_count')
outlier_pct = (outlier_counts / len(df) * 100).round(2).rename('outlier_pct')
outlier_table = pd.concat([outlier_counts, outlier_pct], axis=1).sort_values('iqr_outlier_count', ascending=False)
outlier_table

IQR outliers only show up in three features: `treatment_cost` (2.07% of rows), `medications_count` (1.39%) and `medication_changes_during_stay` (0.42%) — these are the dots beyond the whiskers in the boxplots. Every other numeric feature has 0 flagged outliers under the 1.5×IQR rule.

### 3.3 Categorical/flag feature distributions

In [ ]:
n_cols2 = 3
n_rows2 = int(np.ceil(len(CATEGORICAL_FEATURES) / n_cols2))
fig, axes = plt.subplots(n_rows2, n_cols2, figsize=(15, 4 * n_rows2))
axes = axes.flatten()
for i, col in enumerate(CATEGORICAL_FEATURES):
    vc = df[col].value_counts()
    sns.barplot(x=vc.index.astype(str), y=vc.values, ax=axes[i], palette='viridis')
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=45)
for j in range(len(CATEGORICAL_FEATURES), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

The four text columns (`gender`, `smoking_status`, `discharge_destination`, `insurance_type`) all have fairly even bars across their levels. The binary flags split unevenly instead — `followup_scheduled_flag` skews toward 1, while `icu_admission_flag`, `heart_disease_flag` and `high_risk_medication_flag` skew toward 0 — the same pattern already seen in the Section 1.3 table.

## 4. Bivariate Analysis vs `readmission_flag`

### 4.1 Numeric features grouped by `readmission_flag`

In [ ]:
grouped_stats = df.groupby(TARGET)[NUMERIC_FEATURES].mean().T
grouped_stats.columns = [f'mean_readmit_{c}' for c in grouped_stats.columns]
grouped_stats['mean_diff (1 - 0)'] = grouped_stats['mean_readmit_1'] - grouped_stats['mean_readmit_0']
grouped_stats = grouped_stats.sort_values('mean_diff (1 - 0)', key=abs, ascending=False)
grouped_stats

In [ ]:
n_rows3 = int(np.ceil(len(NUMERIC_FEATURES) / n_cols))
fig, axes = plt.subplots(n_rows3, n_cols, figsize=(15, 4 * n_rows3))
axes = axes.flatten()
for i, col in enumerate(NUMERIC_FEATURES):
    sns.boxplot(data=df, x=TARGET, y=col, ax=axes[i], palette='Set2')
    axes[i].set_title(col)
for j in range(len(NUMERIC_FEATURES), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

`age` has the largest mean gap between classes — 58.2 for readmitted vs 50.8 for not, a +7.4 difference — followed by `length_of_stay_days` (+2.0) and `patient_education_score` (-1.5, lower for readmitted). The remaining numeric features all differ by under 0.6 in their own units, matching how similar their boxplots look between the two `readmission_flag` groups.

### 4.2 Categorical/flag features — readmission rate per category

In [ ]:
readmit_rate_gap = {}
fig, axes = plt.subplots(n_rows2, n_cols2, figsize=(15, 4 * n_rows2))
axes = axes.flatten()
for i, col in enumerate(CATEGORICAL_FEATURES):
    rate = df.groupby(col)[TARGET].mean().sort_values(ascending=False)
    readmit_rate_gap[col] = rate.max() - rate.min()
    sns.barplot(x=rate.index.astype(str), y=rate.values, ax=axes[i], palette='rocket')
    axes[i].set_title(f'{col} (rate gap={readmit_rate_gap[col]:.3f})')
    axes[i].set_ylabel('readmission rate')
    axes[i].tick_params(axis='x', rotation=45)
for j in range(len(CATEGORICAL_FEATURES), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

rate_gap_table = pd.Series(readmit_rate_gap, name='readmission_rate_gap').sort_values(ascending=False).to_frame()
rate_gap_table

Three flags have a much bigger readmission-rate swing between their two levels than everything else: `icu_admission_flag` (gap 0.332 — 63% vs 30%), `followup_scheduled_flag` (gap 0.331 — 58% vs 24%) and `high_risk_medication_flag` (gap 0.224 — 52% vs 30%). Every other feature has a gap under 0.012, i.e. its categories have nearly identical readmission rates.

## 5. Correlation Analysis

### 5.1 Correlation matrix among numeric features

In [ ]:
corr_matrix = df[NUMERIC_FEATURES].corr()
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax, square=True)
ax.set_title('Correlation matrix — numeric features')
plt.tight_layout()
plt.show()

Every off-diagonal cell rounds to 0.00-0.01 — none of these 14 numeric features are linearly correlated with each other in this dataset, so there's no redundant pair to flag from this matrix.

### 5.2 Correlation of numeric features with `readmission_flag`

In [ ]:
flag_cols = ['diabetes_flag', 'hypertension_flag', 'heart_disease_flag', 'icu_admission_flag',
             'emergency_admission_flag', 'high_risk_medication_flag', 'followup_scheduled_flag']
all_numeric_like = NUMERIC_FEATURES + flag_cols

target_corr = df[all_numeric_like + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
target_corr_table = target_corr.rename('corr_with_readmission_flag').to_frame()

fig, ax = plt.subplots(figsize=(8, 8))
sns.barplot(x=target_corr.values, y=target_corr.index, palette='coolwarm', ax=ax)
ax.set_xlabel('Pearson correlation with readmission_flag')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Numeric/flag feature correlation with readmission_flag')
plt.tight_layout()
plt.show()

target_corr_table

Eight features carry a non-trivial correlation with `readmission_flag`: `followup_scheduled_flag` (-0.32), `patient_education_score` (-0.28), `icu_admission_flag` (+0.25), `high_risk_medication_flag` (+0.19), `chronic_conditions_count` (+0.18), `length_of_stay_days` (+0.17), `age` (+0.17) and `previous_admissions_12m` (+0.13). The other 13 all sit under ±0.011 — essentially no linear association with the target.

### 5.3 Chi-square test of independence — categorical features vs `readmission_flag`

In [ ]:
chi2_results = []
for col in CATEGORICAL_FEATURES:
    contingency = pd.crosstab(df[col], df[TARGET])
    chi2, p, dof, expected = stats.chi2_contingency(contingency)
    chi2_results.append({'feature': col, 'chi2_stat': chi2, 'p_value': p, 'dof': dof, 'significant_p<0.05': p < 0.05})

chi2_table = pd.DataFrame(chi2_results).sort_values('chi2_stat', ascending=False).reset_index(drop=True)
chi2_table

Only 4 of the 11 categorical/flag features come back significant at p<0.05: `followup_scheduled_flag` (chi2=5089), `icu_admission_flag` (chi2=3124), `high_risk_medication_flag` (chi2=1786) and `emergency_admission_flag` (chi2=6.0, p=0.014). The other 7 (`insurance_type`, `hypertension_flag`, `discharge_destination`, `diabetes_flag`, `smoking_status`, `gender`, `heart_disease_flag`) all have p > 0.05.

## 6. Data-Driven Feature Ranking Summary

In [ ]:
numeric_rank = target_corr_table.copy()
numeric_rank['abs_metric'] = numeric_rank['corr_with_readmission_flag'].abs()
numeric_rank['metric_type'] = 'pearson_corr_with_target'
numeric_rank = numeric_rank.rename(columns={'corr_with_readmission_flag': 'metric_value'}).reset_index().rename(columns={'index': 'feature'})

categorical_rank = chi2_table[['feature', 'chi2_stat', 'p_value']].copy()
categorical_rank['abs_metric'] = categorical_rank['chi2_stat']
categorical_rank['metric_type'] = 'chi2_stat_with_target'
categorical_rank = categorical_rank.rename(columns={'chi2_stat': 'metric_value'})

numeric_rank['pct_rank'] = numeric_rank['abs_metric'].rank(pct=True)
categorical_rank['pct_rank'] = categorical_rank['abs_metric'].rank(pct=True)

combined = pd.concat([
    numeric_rank[['feature', 'metric_type', 'metric_value', 'pct_rank']],
    categorical_rank[['feature', 'metric_type', 'metric_value', 'pct_rank']].assign(p_value=categorical_rank['p_value'])
], ignore_index=True)

combined = combined.sort_values('pct_rank', ascending=False).reset_index(drop=True)
combined.insert(0, 'overall_rank', combined.index + 1)
combined

Putting both metrics on one percentile scale, `followup_scheduled_flag` takes the top two ranks, followed by `patient_education_score`, `icu_admission_flag` and `high_risk_medication_flag` — the same four features that stood out individually in Sections 5.2/5.3. `heart_disease_flag`, `blood_glucose` and `bmi` land at the very bottom, matching their near-zero correlation/chi-square values above.